Trusted identity → Authorization decision → Data retrieval → Permitted fields → Agent

Notice that E-101 and E-102 are both Northstar adjusters, yet they have different claim permissions.

This is the distinction between:
- RBAC: An adjuster is generally eligible to investigate claims.
- Resource-level access: This particular adjuster is permitted to read this particular claim.

For now, we're implementing explicit resource grants. We'll add role checks later.

In [2]:
from investigation_agent_security_lab.authorization import (
    Principal,
    load_grants,
)
from investigation_agent_security_lab.claims import (
    load_claims,
    get_claim_authorized,
)

from pathlib import Path

cwd = Path.cwd()

PROJECT_ROOT = (
    cwd if (cwd / "pyproject.toml").exists()
    else cwd.parent
)

claims = load_claims(
    PROJECT_ROOT / "data/synthetic/claims.csv"
)
grants = load_grants(
    PROJECT_ROOT / "data/synthetic/access_grants.csv"
)

alice = Principal("E-101", "northstar")
bob = Principal("E-102", "northstar")
charlie = Principal("E-201", "harbor")

In [3]:
scenarios = [
    ("Alice, own claim", alice, "northstar", "C-1042"),
    ("Alice, unauthorized claim", alice, "northstar", "C-1043"),
    ("Bob, own claim", bob, "northstar", "C-1043"),
    ("Alice, other tenant", alice, "harbor", "C-1042"),
    ("Charlie, own claim", charlie, "harbor", "C-1042"),
]

for label, principal, tenant, claim_id in scenarios:
    try:
        record = get_claim_authorized(
            claims, grants, principal, tenant, claim_id
        )
        outcome = "ALLOWED" if record else "NOT FOUND"
    except PermissionError:
        outcome = "DENIED"

    print(f"{label}: {outcome}")

Alice, own claim: ALLOWED
Alice, unauthorized claim: DENIED
Bob, own claim: ALLOWED
Alice, other tenant: DENIED
Charlie, own claim: ALLOWED


This demonstrates something important: authorization isn't simply whether a user belongs to a tenant. It's whether that verified user can perform a particular operation on a particular resource.